# C1 Sensor Data Exploration

This notebook profiles the locally downloaded dataset before implementing the C1 validation service.

The dataset must be placed in `data/raw/`. It is intentionally excluded from Git. The final year is reserved exclusively for testing; this notebook only inspects the data and demonstrates the chronological split.

In [1]:
from pathlib import Path

import pandas as pd

data_dirs = [Path("data/raw"), Path("../data/raw")]
candidates = sorted(
    path
    for data_dir in data_dirs
    for path in [*data_dir.glob("*.csv"), *data_dir.glob("*.xlsx"), *data_dir.glob("*.xls")]
)

if not candidates:
    raise FileNotFoundError(
        "No CSV or Excel file found in data/raw. Download the dataset there before running this notebook."
    )

DATA_PATH = candidates[0]
print(f"Using: {DATA_PATH}")

Using: ../data/raw/sensor_data.csv


In [2]:
if DATA_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(DATA_PATH)
else:
    df = pd.read_excel(DATA_PATH)

df.head()

,timestamp,machine_id,operating_mode,temperature,vibration,pressure,rpm,power,anomaly,anomaly_type,failure_probability_gt,failure_in_7_days,remaining_useful_life_hours,failure_type,important_features,maintenance_action,priority,concept_drift_active
0,"2012-01-14 00:00:00,M001,HIGH_LOAD,46.07525961...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"2012-01-14 00:00:00,M002,NORMAL,46.58425922252...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"2012-01-14 00:00:00,M003,IDLE,40.5464513381037...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2012-01-14 00:00:00,M004,NORMAL,48.470209,2.953438,97.447963,1515.488150,211.998125,False,NaN,0.374622,False,625.0,cooling_failure,temperature,repair_cooling_system,low,False
4,2012-01-14 00:00:00,M005,NORMAL,39.141233,1.222667,99.561297,1504.955537,209.234533,False,NaN,0.049056,False,950.0,cooling_failure,temperature,repair_cooling_system,low,False


## Basic profile

In [ ]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
display(df.dtypes.to_frame("dtype"))
display(df.describe(include="all").T)

In [ ]:
missing = (
    df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_rate=lambda table: table["missing_count"] / len(df))
    .sort_values("missing_count", ascending=False)
)
display(missing)

## Data quality and coverage

In [ ]:
required_columns = {
    "timestamp",
    "machine_id",
    "operating_mode",
    "temperature",
    "vibration",
    "pressure",
    "rpm",
    "power",
}
print("Missing required columns:", sorted(required_columns - set(df.columns)))
print("Duplicate rows:", int(df.duplicated().sum()))

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
print("Invalid timestamps:", int(df["timestamp"].isna().sum()))
print("Time range:", df["timestamp"].min(), "to", df["timestamp"].max())

In [ ]:
for column in ["machine_id", "operating_mode", "anomaly_type", "failure_type", "maintenance_action", "priority"]:
    if column in df:
        print(f"\n{column}")
        display(df[column].value_counts(dropna=False).to_frame("count"))

## Chronological split

The first two years are available for training and validation. The final year is held out for testing and must not be used for model selection.

In [ ]:
valid_df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
start = valid_df["timestamp"].min()
test_start = valid_df["timestamp"].max() - pd.DateOffset(years=1)
train_validation = valid_df[valid_df["timestamp"] < test_start].copy()
test = valid_df[valid_df["timestamp"] >= test_start].copy()

split_summary = pd.DataFrame(
    {
        "partition": ["train_validation", "test"],
        "start": [train_validation["timestamp"].min(), test["timestamp"].min()],
        "end": [train_validation["timestamp"].max(), test["timestamp"].max()],
        "rows": [len(train_validation), len(test)],
    }
)
display(split_summary)
assert train_validation["timestamp"].max() < test["timestamp"].min()


## Next steps

- Confirm the actual column names against the assignment dataset.
- Decide which sensor ranges and categorical values are valid.
- Document anomalies and missing-value patterns.
- Implement the validated schema and processing rules in the C1 service.
